# Macao's Demographic Transition Over 25 Years (1999–2024): A Data-Driven Analysis via Interactive Dashboard

**Course:** CISC7201 Data Science Programming

**Group:** BH

**Date:** December 14, 2025

### Project Overview

Our report documents a complete, end-to-end data science workflow designed to analyze the profound demographic shifts in the Macao Special Administrative Region (SAR) from **1999 to 2024**. Moving beyond standard descriptive statistics, our project synthesizes data preprocessing, exploratory analysis, and interactive visualization to reveal the hidden structural challenges of one of the world's most densely populated regions.

This research relies on official data from the **Statistics and Census Service (DSEC)**. We processed information from the **Time Series Database** and annual **Statistical Yearbooks** to create a unified dataset tracking population structure and migration. Instead of presenting static charts, we developed an interactive dashboard using **Streamlit**. This platform integrates **Plotly** visualizations to guide users through Macao’s demographic shifts. By converting raw statistics into an explorable tool, the project allows for a practical investigation of the **'Growth-Based Aging' Paradox**, helping users identify the structural changes hidden behind Macao’s aggregate growth.

## 1. Motivation and Team Background

### 1.1 Motivation

**The Context: A Unique Demographic Laboratory**

Since its handover to China in 1999, Macao has served as a unique laboratory for demographic study. As a micro-economy heavily reliant on gaming and tourism, the region has experienced explosive population growth driven by the liberalization of the gaming industry in 2002. However, this aggregate growth conceals a critical structural vulnerability.

**The Problem: The "Growth-Based Aging" Paradox**

Traditional static reports often fail to capture the nuance of Macao’s demographic reality. On the surface, the population appears robust and growing. However, our preliminary research identified a phenomenon we term **"Growth-Based Aging."** The massive influx of non-resident workers (who are typically young and working-age) has historically diluted the aging statistics, creating a **"statistical shield"** that hides the rapid aging of the local resident population. Recent external shocks, particularly the COVID-19 pandemic, exposed the fragility of this model when the non-resident labor force contracted, leaving the aging local structure exposed.

**The Solution: Why a Data Science Approach?**

The motivation of our project is to move beyond the limitations of static PDF yearbooks. By applying data science techniques—specifically **data engineering** to harmonize disparate sources and **interactive visualization** to visualize and animate changes over time—we aim to answer critical questions that standard tables cannot:
1.  **Structural Masking:** To what extent does the non-resident workforce conceal the true dependency burden of the local society?
2.  **Spatial Saturation & Carrying Capacity:** With major reclamation zones (e.g., Cotai) often serving non-residential functions, how does the density evolution within the traditional districts (Peninsula, Taipa, and Coloane) reveal the genuine physical constraints of Macao's living environment?
3.  **Future Trajectories:** Is the current demographic model sustainable through 2035 under varying economic scenarios?

This project answers these questions by transforming raw administrative data into an interactive evidence base, providing the clarity needed to navigate Macao’s transition from a period of rapid expansion to one of structural maturity.


### 1.2 Team Background
Our team combines expertise in data engineering, visualization, and domain-specific socioeconomic research. This diversity allowed us to tackle the full data science lifecycle, from scraping official government databases to deploying a reactive web application.

*   **Ma Man Hin (MC565141):** *Project Lead & Full-Stack Developer.* Proposed the project concept and architected the Streamlit dashboard. Responsible for the data preprocessing pipeline and the interactive population pyramid visualizations.
*   **Leong Tin Long (MC564437):**
*   **Lei Sam Tat (MC566546):** *
*   **XiaoJiang Li (MC564588):** contributions to the project included assisting with report writing, code review, and video presentations.

## 2. Data Collection

To ensure analytical rigor, we prioritized primary administrative records over generalized international datasets. We compiled our data exclusively from the authoritative archives of the **Statistics and Census Service (DSEC)**. Although global repositories like the World Bank were evaluated during the preliminary research phase, they were ultimately excluded due to significant limitations in data dimensionality and completeness. Consequently, DSEC was selected as the sole primary source, offering the necessary depth to accurately capture Macao's demographic context.

The data collection process involved synthesizing **two distinct raw inputs** to serve different analytical needs:

1.  **Time-Series Database:** Longitudinal data covering total population, vital statistics (birth/death rates), regional density, and migration flows was retrieved directly from the DSEC official online database.
2.  **Yearbook-Derived Structural Data:** To construct the high-fidelity population pyramids, we utilized the "End-Year Total Population by Age Group" sections from the annual Statistical Yearbooks. As this data was not available in time-series database, we **manually extracted the granular age-group and gender counts and compiled them into a wide-format dataset**. This manual curation was essential to ensure the structural accuracy required for the cohort analysis visualizations.

To facilitate geospatial visualization, we retrieved administrative boundary geometries via an **OGC Web Feature Service (WFS) hosted by the Stanford Digital Repository (EarthWorks)**, further complemented by **OpenStreetMap** data to ensure topological accuracy for the density analysis across Macao Peninsula, Taipa, and Coloane.


In [3]:
# Print dataset shapes and column names
from pathlib import Path
import pandas as pd
DATA_DIR = Path('data/processed')
datasets = {
    'demographics': 'macao_demographics_1999_2024.csv',
    'pyramid': 'population_pyramid_data.csv',
    'pyramid_pct': 'population_pyramid_data_percentage.csv',
}
for name, fname in datasets.items():
    p = DATA_DIR / fname
    if p.exists():
        df = pd.read_csv(p)
        print(f'{fname} -> shape: {df.shape}')
        print('Columns:', df.columns.tolist())
# Print the Age Group column of pyramid dataset
        if 'Age Group' in df.columns:
            age_groups = df['Age Group'].dropna().unique().tolist()
            print(f'Unique Age Groups ({fname}):', age_groups)
    else:
        print(f'File not found: {p}')


macao_demographics_1999_2024.csv -> shape: (26, 27)
Columns: ['Year', 'Total population', 'Male', 'Female', 'Below Age 15', 'Age 15-24', 'Age 25-34', 'Age 35-44', 'Age 45-54', 'Age 55-64', 'Age 65 and above', 'Annual growth rate', 'Non-resident workers total', 'Chinese mainland', 'Philippines', 'Vietnam', 'Construction', 'Hotels, Restaurants & Similar Activities', 'Recreational, Cultural, Gaming & Other Services', 'New arrivals from Chinese mainland with one-way permit', 'Crude birth rate', 'Crude mortality rate', 'Rate of natural increase', 'Population density', 'Macao Peninsula a', 'Taipa a', 'Coloane a']
population_pyramid_data.csv -> shape: (16, 53)
Columns: ['Age Group', 'M_1999', 'F_1999', 'M_2000', 'F_2000', 'M_2001', 'F_2001', 'M_2002', 'F_2002', 'M_2003', 'F_2003', 'M_2004', 'F_2004', 'M_2005', 'F_2005', 'M_2006', 'F_2006', 'M_2007', 'F_2007', 'M_2008', 'F_2008', 'M_2009', 'F_2009', 'M_2010', 'F_2010', 'M_2011', 'F_2011', 'M_2012', 'F_2012', 'M_2013', 'F_2013', 'M_2014', 'F_20

## 3. Data Engineering, Cleaning, and Management

To ensure the integrity and reproducibility of our analysis, we engineered a rigorous data processing pipeline using **Python**. The primary objective was to harmonize the heterogeneous raw inputs obtained from DSEC—ranging from formatted Excel reports to raw database extracts—into a unified, machine-readable schema optimized for visualization.

### 3.1 Data Preprocessing Pipeline
The raw datasets presented distinct structural challenges, utilizing hierarchical headers intended for human readability rather than computational analysis. We leveraged the **pandas** and **numpy** libraries to implement a systematic cleaning workflow.

#### I. Time-Series Dataset Processing
The longitudinal data required a fully computational transformation to standardize dimensions across the 1999–2024 timeline:

*   **Programmatic Header Flattening:** Source files frequently employed multi-level hierarchical indexing (e.g., "By Gender" grouped above "Male/Female"). We developed parsing logic to collapse these stacked headers into unambiguous, single-level semantic identifiers. This step was crucial for enabling consistent variable referencing in the dashboard’s codebase.
*   **Handling Censored Data (Imputation):** A specific challenge involved non-numeric annotations, particularly the "0#" symbol used by DSEC to denote a value falling below the reporting threshold. To preserve the continuity of growth rates without introducing statistical noise, we applied a deterministic imputation strategy: these censored values were coerced to `0.05` (representing half of the reporting unit). This approach allows for mathematical modeling while acknowledging the minimal magnitude of the data point.
*   **Type Coercion and Sanitization:** The pipeline enforced strict numeric typing across all analytical columns. Non-numeric placeholders indicating missing data (e.g., "~") were systematically converted to `NaN` (Not a Number) to ensure they were correctly excluded from aggregation logic and clearly flagged in the visualization layer.



In [ ]:
# Preprocessing module for DSEC time-series CSV -> cleaned CSV.
from __future__ import annotations

import os
import re
from typing import Optional

import numpy as np
import pandas as pd

# -----------------------
# I. Paths & Configuration
# -----------------------
HERE = os.path.dirname(__file__)
INFILE = os.path.join(HERE, 'dsec_dataset.csv')
OUTDIR = os.path.join(os.path.dirname(HERE), 'processed')
OUTFILE = os.path.join(OUTDIR, 'macao_demographics_1999_2024.csv')

# Expected column layout inferred from the raw dataset.
original_columns = [
    'Year',
    'Total population',          # '000
    'Male',                      # '000
    'Female',                    # '000
    'Total population (dup)',    # duplicate — will be dropped
    'Below Age 15',              # '000
    'Age 15-24',
    'Age 25-34',
    'Age 35-44',
    'Age 45-54',
    'Age 55-64',
    'Age 65 and above',          # '000
    'Annual growth rate',        # %
    'Non-resident workers total',# No.
    'Chinese mainland',          # No.
    'Philippines',
    'Vietnam',
    'Construction',              # No.
    'Hotels, Restaurants & Similar Activities',
    'Recreational, Cultural, Gaming & Other Services',
    'New arrivals from Chinese mainland with one-way permit',  # No.
    'Crude birth rate',          # o/oo
    'Crude mortality rate',
    'Rate of natural increase',
    'Population density',        # '000/km²
    'Macao Peninsula a',         # '000/km²
    'Taipa a',
    'Coloane a',
    'Note'                       # annotation column (e.g., ~, 0#)
]

# -----------------------
# II. Helper functions
# -----------------------

def find_data_start_row(filepath: str) -> Optional[int]:
    """Detect first row index where the first column looks like a 4-digit year."""
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            first_cell = line.split(',')[0].strip().strip('"')
            if re.match(r'^\d{4}$', first_cell):
                return i
    return None


def clean_value(x):
    """Sanitize cell values:
       - Map '~', '0#', empty or annotation-only values to NaN
       - Remove enclosing quotes and thousands separators (commas)
       - Preserve numeric-like strings for later coercion
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == '~' or s == '0#' or s == '':
        return np.nan
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        s = s[1:-1]
    s = s.replace(',', '')
    if re.match(r'^[^0-9.+-]+$', s):
        return np.nan
    return s


# -----------------------
# III. Processing logic
# -----------------------

def preprocess(infile: str = INFILE, outfile: str = OUTFILE) -> pd.DataFrame:
    """Main preprocessing routine:
       - Detect header row
       - Read CSV robustly into a DataFrame
       - Normalize columns and types
       - Handle specific annotation conversions and fill rules
       - Save processed CSV
    """
    if not os.path.exists(infile):
        raise FileNotFoundError(f"Input file not found: {infile}")

    start_row = find_data_start_row(infile)
    if start_row is None:
        raise RuntimeError('Could not find the data start row by year detection')

    # Read the CSV starting at the detected row.
    df = pd.read_csv(infile, skiprows=start_row, header=None, encoding='utf-8', engine='python')

    # Ensure df has the expected number of columns (pad or trim as needed)
    ncols_needed = len(original_columns)
    if df.shape[1] < ncols_needed:
        for c in range(df.shape[1], ncols_needed):
            df[c] = np.nan
    elif df.shape[1] > ncols_needed:
        df = df.iloc[:, :ncols_needed]

    df.columns = original_columns[: df.shape[1]]

    # Drop duplicate or purely-annotation columns
    df = df.drop(columns=['Total population (dup)', 'Note'], errors='ignore')

    # Apply cleaning function to all cells
    df = df.applymap(clean_value)

    # Convert Year to numeric and coerce non-numeric to NaN
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

    # Cast all other columns to numeric where possible
    for col in df.columns:
        if col == 'Year':
            continue
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Dataset-specific imputation convention for Annual growth rate
    if 'Annual growth rate' in df.columns:
        df['Annual growth rate'] = df['Annual growth rate'].fillna(0.05)

    # Filter to target analysis window
    df = df[(df['Year'] >= 1999) & (df['Year'] <= 2024)].copy()

    # Ensure output directory exists
    outdir = os.path.dirname(outfile)
    os.makedirs(outdir, exist_ok=True)

    # Persist the cleaned CSV
    df.to_csv(outfile, index=False)
    return df


# -----------------------
# IV. CLI Entrypoint
# -----------------------

if __name__ == '__main__':
    df_out = preprocess()  # use INFILE/OUTFILE defaults
    print(f'Cleaned file written to: {OUTFILE}')
    print(f'Dataframe shape: {df_out.shape}')

#### II. Population Pyramid Structuring

The preparation of the population pyramid dataset necessitated a hybrid workflow combining manual curation with algorithmic transformation.

*   **Manual Categorical Aggregation** Prior to computational processing, we manually aggregated the upper age cohorts—specifically "75–79," "80–84," and "≧85"—into a single "75+" category within the source Excel file. This pre-processing step was essential to mitigate data sparsity artifacts in the senior demographic bands and ensure visual consistency across the timeline.
*   **Transformation to Long Format & Preprocessing:** The curated data (wide-format with year columns) was reshaped into a tidy long format ([Age Group, Year, Sex, Count]) using pandas `melt`; numeric columns were coerced (commas stripped) and validated — the source pyramid CSVs contain no missing values. For robustness, any non-numeric entries would be coerced to `NaN` and the plotting pipeline fills such cases with `0` for visualization. Normalized percentage shares were prepared for the normalized mode. `Age Group` was set as an ordered categorical to preserve plotting order, male counts were negated to support mirrored bar rendering, and a `Year`/`frame` column was included for Plotly’s frame-based animation. The transformed dataset was validated by re-aggregating totals and cross-checking with DSEC figures.


In [ ]:
# Pyramid preprocessing 

from __future__ import annotations

import os
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd


def _resolve_csv_path(csv_path: str) -> Path:
    """Resolve CSV path relative to common working directories."""
    path = Path(csv_path)
    if path.exists():
        return path
    alt = Path("../") / path
    if alt.exists():
        return alt
    return path


def _read_pyramid_csv(path_str: str) -> pd.DataFrame:
    """Read a pyramid CSV as strings (no caching)."""
    return pd.read_csv(path_str, dtype=str, keep_default_na=False)


def load_pyramid_csv(csv_path: str) -> pd.DataFrame:
    """Load a pyramid CSV.

    - Reads all columns as strings first (robust to formatting)
    - Coerces non 'Age Group' columns to numeric (stripping commas)
    - Drops fully empty rows
    """
    path = _resolve_csv_path(csv_path)
    mtime_ns = path.stat().st_mtime_ns if path.exists() else 0
    df = _read_pyramid_csv(str(path)).copy()
    df = df.replace({"": pd.NA}).dropna(how="all")

    for col in df.columns:
        if col != "Age Group":
            df[col] = pd.to_numeric(
                df[col].str.replace(",", "", regex=False).str.strip(),
                errors="coerce",
            )

    return df


def pivot_pyramid_df(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """Convert a wide pyramid dataframe into a long form suitable for plotting.

    Returns:
      - long: DataFrame with columns ['Age Group', 'Year', 'Sex', 'Population', 'Population_Neg', 'Population_Abs']
      - age_order: list of Age Group strings preserving source order
    """
    value_cols = [c for c in df.columns if c != "Age Group"]

    long = df.melt(
        id_vars=["Age Group"],
        value_vars=value_cols,
        var_name="Year_Sex",
        value_name="Population",
    )

    # Expect column names like "M_1999" or "F_1999"
    long["Sex"] = long["Year_Sex"].str[0].map({"M": "Male", "F": "Female"})
    long["Year"] = long["Year_Sex"].str.split("_").str[1].astype(int)

    long["Population"] = pd.to_numeric(long["Population"], errors="coerce").fillna(0.0)
    # Mirror male population to negative for left-side plotting
    long["Population_Neg"] = np.where(long["Sex"] == "Male", -long["Population"], long["Population"])
    long["Population_Abs"] = long["Population"].abs()

    age_order = list(df["Age Group"].dropna().unique())
    return long, age_order


def preprocess_pyramid(csv_path: str) -> Tuple[pd.DataFrame, List[str]]:
    """Wrapper: load and pivot a pyramid CSV."""
    df = load_pyramid_csv(csv_path)
    return pivot_pyramid_df(df)


if __name__ == "__main__":
    sample_path = os.path.join(os.path.dirname(__file__), "..", "processed", "population_pyramid_data.csv")
    if Path(sample_path).exists():
        long_df, order = preprocess_pyramid(sample_path)
        print("Loaded pyramid -> long shape:", long_df.shape)
        print("Age groups:", order)
        print("Years:", sorted(long_df["Year"].unique()))
    else:
        print(f"No sample pyramid CSV found at: {sample_path}")

#### III. Geospatial Logic
*   **Vector Processing:** We utilized `geopandas` to align administrative vector geometries sourced from the Stanford Digital Repository (EarthWorks) WFS and OpenStreetMap data. Layers were standardized to a common projected CRS suitable for accurate area calculations, and topology fixes (e.g., `buffer(0)`, `unary_union`, `dissolve`) were applied to remove slivers and gaps.
*   **Topology & Cleanup:** We repaired and validated geometries using `shapely` operations (e.g., `buffer(0)`, `unary_union`) and dissolved adjacent administrative segments where necessary to create clean region polygons for aggregation.
*   **Spatial Subtraction & Masking:** A spatial `difference` step was applied to remove overlapping island polygons, explicit non-residential areas, and water bodies from the administrative layers. This subtraction ensured that area calculations reflect the physically inhabitable surface (km²) used for density calculations.
*   **Attribute Join & Density Calculation:** Cleaned vector geometries were joined with tabular population data (sourced from DSEC) via a keyed attribute join or spatial join where necessary. For years where DSEC provides region-level density figures, those columns were used; for earlier records lacking region-level breakdowns (e.g., 1999–2006), the overall population density was used as a fallback. We computed polygon areas in square meters using the projected CRS, converted them to km², and derived `Persons/km²` by dividing population counts by the computed area for each year snapshot used in the choropleth visualization.
*   **Validation & Reproducibility:** Results were validated by cross-checking computed areas against official totals and by visually inspecting choropleth outputs. The cleaned, reprojection-consistent geometries and derived density metrics were persisted in the `processed/` folder to support reproducible mapping and time-series animation in the Streamlit dashboard.

In [ ]:
def load_shapefile() -> GeoDataFrame:
    shp_path = next((p for p in SHP_PATHS if p.exists()), SHP_PATHS[0])
    if not shp_path.exists():
        raise FileNotFoundError(f"Shapefile not found at {shp_path}")
    gdf = gpd.read_file(shp_path)
    gdf = gdf[~gdf.geometry.isna() & ~gdf.geometry.is_empty]
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    return gdf

def load_geojson() -> GeoDataFrame:
    if not GEOJSON_PATH.exists():
        raise FileNotFoundError(f"GeoJSON not found at {GEOJSON_PATH}")
    gdf = gpd.read_file(GEOJSON_PATH)
    # filter to Macau / Taipa / Coloane entries
    macau_filter = (gdf['name'] == '澳門 Macau') | (gdf['name:en'] == 'Macau')
    taipa_filter = (gdf['name'] == '氹仔 Taipa') | (gdf['name:en'] == 'Taipa')
    coloane_filter = (gdf['name'] == '路環 Coloane') | (gdf['name:en'] == 'Coloane')
    gdf = gdf[macau_filter | taipa_filter | coloane_filter]
    gdf = gdf[~gdf.geometry.isna() & ~gdf.geometry.is_empty]
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    return gdf

@lru_cache(maxsize=1)
def prepare_geospatial_data() -> Tuple[GeoDataFrame, Dict]:
    # load base shapefile & region polygons
    macau_gdf = load_shapefile()
    regions_gdf = load_geojson()

    # extract Taipa & Coloane geometries from regions_gdf
    taipa_geom = regions_gdf.loc[(regions_gdf['name']=='氹仔 Taipa') | (regions_gdf['name:en']=='Taipa')].geometry.iloc[0]
    coloane_geom = regions_gdf.loc[(regions_gdf['name']=='路環 Coloane') | (regions_gdf['name:en']=='Coloane')].geometry.iloc[0]

    macau_boundary = macau_gdf.iloc[0].geometry

    # Clean geometries and derive peninsula by subtracting islands
    macau_boundary_clean = macau_boundary.buffer(0)
    taipa_geom_clean = taipa_geom.buffer(0)
    coloane_geom_clean = coloane_geom.buffer(0)
    peninsula_geom = macau_boundary_clean.difference(taipa_geom_clean).difference(coloane_geom_clean)

    # If MultiPolygon from difference, choose largest significant part as peninsula
    if peninsula_geom.geom_type == 'MultiPolygon':
        parts = list(peninsula_geom.geoms)
        total_area = sum(p.area for p in parts)
        significant_parts = [p for p in parts if p.area > total_area * 0.01]
        peninsula_geom = max(significant_parts, key=lambda g: g.area)

    # Build combined GeoDataFrame: Macau Base, Macao Peninsula, Taipa, Coloane
    records = [
        {'region_name': 'Macau Base', 'geometry': macau_boundary, 'is_base_layer': True},
        {'region_name': 'Macao Peninsula', 'geometry': peninsula_geom, 'is_base_layer': False},
        {'region_name': 'Taipa', 'geometry': taipa_geom, 'is_base_layer': False},
        {'region_name': 'Coloane', 'geometry': coloane_geom, 'is_base_layer': False},
    ]
    gdf = gpd.GeoDataFrame(records, geometry='geometry', crs=macau_gdf.crs)

    # Export GeoJSON, falling back to manual mapping if needed
    try:
        geojson = json.loads(gdf.to_json())
    except Exception:
        from shapely.geometry import mapping
        features = []
        for _, row in gdf.iterrows():
            geom_obj = row.geometry
            if geom_obj is None or geom_obj.is_empty:
                continue
            features.append({
                "type": "Feature",
                "geometry": mapping(geom_obj),
                "properties": {"region_name": row["region_name"]}
            })
        geojson = {"type": "FeatureCollection", "features": features}

    return gdf, geojson

### 3.2 Data Management & Reproducibility
*   **Server-Side Caching:** To optimize I/O performance, processed datasets are stored in memory using Streamlit’s caching mechanisms, preventing redundant computation during user interaction.
*   **Exception Handling:** We implemented specific logic to handle historical data gaps (common in pre-2007 records) by rendering fallback messages or "N/A" indicators rather than breaking the visualization pipeline.

*(Note: Key Python libraries used include `pandas`, `numpy`, `streamlit`, `plotly.graph_objects`, and `plotly.express`.)*


In [ ]:
# Example 1: LRU cache example

# Read the CSV as strings and cache the result keyed by path + mtime.
@lru_cache(maxsize=8)
def _read_pyramid_csv_cached(path_str: str, mtime_ns: int) -> pd.DataFrame:
    """Read a pyramid CSV as strings and cache based on file mtime."""
    return pd.read_csv(path_str, dtype=str, keep_default_na=False)

def load_pyramid_csv(csv_path: str) -> pd.DataFrame:
    """Load a pyramid CSV. Uses LRU-cached reader where available."""
    path = _resolve_csv_path(csv_path)
    mtime_ns = path.stat().st_mtime_ns if path.exists() else 0
    # Use cached loader which is keyed by mtime to reload when file changes
    df = _read_pyramid_csv_cached(str(path), mtime_ns).copy()
    df = df.replace({"": pd.NA}).dropna(how="all")


# Example 2: Streamlit caching example

# Demonstrates how to cache the processed long-form pyramid dataset using Streamlit.
@_st_cache(ttl=3600)  # cached for 1 hour; no-op if streamlit not installed
def preprocess_pyramid_cached(csv_path: str) -> Tuple[pd.DataFrame, List[str]]:
    """Streamlit-cached wrapper of preprocess_pyramid."""
    return preprocess_pyramid(csv_path)

if __name__ == "__main__":
    sample_path = os.path.join(os.path.dirname(__file__), "..", "processed", "population_pyramid_data.csv")
    if Path(sample_path).exists():
        # Example usage: LRU-based load & Streamlit cached prep
        long_df, order = preprocess_pyramid_cached(sample_path)
        print("Loaded pyramid -> long shape:", long_df.shape)
        print("Age groups:", order)
        print("Years:", sorted(long_df["Year"].unique()))
    else:
        print(f"No sample pyramid CSV found at: {sample_path}")

        
# Example 3: Exception Handling for Missing Data
def get_region_density_for_year(df: pd.DataFrame, region_col: str, year: int):
    """Return region density for a given year with graceful fallback 'N/A'.

    - If a region-specific density value is missing (common in pre-2007 data),
      attempt to fall back to the overall 'Population density' column.
    - Always return 'N/A' rather than raising, to prevent breaking the dashboard.
    """
    try:
        if region_col not in df.columns:
            return "N/A"
        row = df.loc[df["Year"] == year]
        if row.empty:
            return "N/A"
        val = row[region_col].iloc[0]
        if pd.isna(val):
            # Use overall population density for early years where region breakdowns are unavailable
            if "Population density" in df.columns:
                overall = row["Population density"].iloc[0]
                return overall if not pd.isna(overall) else "N/A"
            return "N/A"
        return val
    except Exception:
        return "N/A"
        


## 4. Data Analysis: Learning, Analytics, Visualization

### 4.1 Interactive Dashboard Architecture

To transition from static statistical reporting to a structured analytical product, we utilized **Streamlit** as the application framework. In our architecture, Streamlit functions not as the visualization engine, but as the **orchestration layer** that organizes our Python analysis into a cohesive, user-friendly interface. Its primary contributions to the project structure are:

*   **Modular Layout and Navigation:** Streamlit acts as the high-level container that encapsulates our distinct analytical modules—*Overview, Population Pyramids, and Demographic Analysis*—into separate, navigable views. This allows us to organize the narrative logically, guiding the user from a macro-level health check to granular cohort inspections without code exposure.
*   **Session State Management (Overview Module):** The power of Streamlit is most evident in the **Overview** page. We implemented a global "Session State" to manage the temporal context. A master year-slider (1999–2024) is bound to this state; when a user adjusts the year, Streamlit triggers a reactive update across the entire layout. This synchronously refreshes the KPI metric cards, updates the specific density values fed into the map, and recalculates the gender distribution, ensuring that all distinct elements reflect the same slice of time instantaneously.
*   **Plotly Integration:** While Streamlit manages the container, the visualization logic is handled by **Plotly**. Streamlit renders these interactive Plotly objects, preserving their native features (such as zooming, panning, and tooltips) within the web layout. This separation of concerns—Streamlit for structure/state, Plotly for rendering—ensures the dashboard is both responsive and analytically rich.


In [ ]:
# Minimal Streamlit app demonstrating:
#  - Modular layout (Overview / Pyramids / Demographics)
#  - Global session-state year slider
#  - Plotly integration 

import streamlit as st
import plotly.graph_objects as go
import pandas as pd

# local imports from project modules 
from graphs.choropleth_builder import prepare_geospatial_data, build_choropleth_figure
from graphs.pyramid_builder import preprocess_pyramid_cached  

# Initialize session state
if "year" not in st.session_state:
    st.session_state.year = 2024

# Sidebar: page nav + global year control
st.sidebar.title("Navigation")
page = st.sidebar.radio("Page", ["Overview", "Population Pyramids", "Demographic Analysis"])

# Global year slider bound to session_state
st.sidebar.slider("Year", min_value=1999, max_value=2024, value=st.session_state.year, key="year")

# Load required data once (cached within modules if available)
demographics_df = pd.read_csv("data/processed/macao_demographics_1999_2024.csv")
regions_gdf, geojson = prepare_geospatial_data()

def render_overview():
    year = st.session_state.year
    st.header(f"Overview — {year}")

    # KPI row (compact)
    col1, col2, col3 = st.columns(3)
    total = demographics_df.loc[demographics_df["Year"] == year, "Total population"]
    col1.metric("Total population", f"{int(total.values[0]) if not total.empty else 'N/A'}")

    # Demonstrate region density fallback using helper
    mp_density = get_region_density_for_year(demographics_df, "Macao Peninsula a", year)
    col2.metric("Macao Peninsula density", mp_density)

    taipa_density = get_region_density_for_year(demographics_df, "Taipa a", year)
    col3.metric("Taipa density", taipa_density)

    # Choropleth built by project function (Plotly)
    fig = build_choropleth_figure(year, demographics_df, regions_gdf, geojson, global_vmin=0, global_vmax=60000)
    if fig:
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Choropleth not available for selected year.")

# Page router
if page == "Overview":
    render_overview()
elif page == "Population Pyramids":
    render_pyramids()
else:
    render_demographics()

See more in 

[Open streamlit_app.py](streamlit_app.py) (Streamlit dashboard)

[Open config.py](config.py) (Dashboard configuration)

[Open data_loader.py](modules/data_loader.py) (Data loading)

[Open ui_components.py](modules/ui_components.py) (UI components)



### 4.2 Overview: Demographic Health Monitoring

**Objective:** To provide a multi-dimensional synthesis of Macao's demographic health, functioning as the dashboard's central command interface. This view allows users to rapidly assess core population dynamics—from regional density to vital statistics—before conducting granular cohort analysis.

**Visualization Description:** The interface is systematically organized into five analytical modules:
*   **Key Demographic Metrics & Gender Distribution:** The dashboard initiates with a high-level assessment panel.
    *   **KPI Grid:** A responsive layout of eight metric cards presenting core indicators (Total Population, Density, Growth Rates, Vital Statistics, Dependency Ratio, and Median Age). Each card prioritizes the primary value alongside a color-coded Year-over-Year (YoY) change indicator to facilitate rapid trend diagnosis.
    *   **Gender Module:** A dedicated section juxtaposing male and female population shares. It aggregates absolute counts, relative percentages, and YoY growth badges, balancing graphical cues with precise numeric values.


*   **Geospatial Density Map:**
    *   **Choropleth Map:** Population density across the Macao Peninsula, Taipa, and Coloane is visualized using a continuous color scale (0–60,000 persons/km², light yellow to red). The layer was constructed by synthesizing official WFS boundary data with OpenStreetMap island geometries.
    *   **Regional Metrics:** Adjacent to the map, district-specific KPI cards report density figures and YoY changes for each region. These components mirror the visual language of the main KPI grid and include contextual alerts (e.g., "N/A") when historical regional data is unavailable.


*   **Structural Age Composition:**
    *   **Macro Age Buckets:** A donut chart summarizes the population into broad categories (0–14, 15–64, 65+), featuring a central total population annotation to maintain context between distribution and magnitude.
    *   **Temporal Heatmap:** A vertical heatmap renders the evolution of fine-grained age groups across the 25-year time series.
    *   **Dependency Ratios:** A composite chart combines stacked bars (distinguishing child vs. senior dependency) with a line overlay for the total dependency ratio, allowing users to decouple the constituent drivers of dependency from the aggregate trend.


*   **Labor Market overview**
    *   **Non-Resident Worker Dynamics:** Triangulates workforce data using a pie chart for origin composition (e.g., Mainland China, Philippines) and a horizontal bar chart for YoY percentage shifts.

*   **Summary Report:** 

    An automated text-based report aggregates the session’s key findings into four thematic quadrants—Population, Age Structure, Regional Density, and YoY Changes—providing a concise executive summary.

**Dashboard Features:**
*   **Global Temporal Control:** A master slider (spanning 1999–2024) is bound to the application’s global session state. Adjusting this parameter triggers a reactive update across all active visualizations (e.g., map coloring, KPI values, and age buckets).
*   **Synchronous Interactivity:** The system prioritizes user interaction through unified hover tooltips. Hovering over a specific year in the trend chart, for instance, aligns the data display across related metrics.
*   **Robust Exception Handling:** To ensure transparency regarding historical data limitations, the interface implements contextual logic. For example, pre-2007 density data triggers specific "N/A" indicators on the regional KPI cards rather than breaking the visualization, ensuring the system degrades gracefully.


[Open overview.py](sections/overview.py) (Overview section)

[Open choropleth_builder.py](graphs/choropleth_builder.py) (Choropleth)


### 4.3 Structural Cohort Analysis: The Population Pyramids

**Objective:** To decouple raw population growth from compositional aging and reveal the "dual narrative" of labor migration versus local demographic transition.

**Visualization Description:** We designed a **Dual-Mode Animated Population Pyramid** that facilitates cohort analysis through two distinct perspectives:
*   **Absolute Numbers Mode:** Visualizes raw population counts (in thousands) via a mirrored bar chart. The x-axis maps males to negative values (left-oriented) and females to positive values (right-oriented), revealing the total volume of specific age groups.
*   **Percentage Mode:** Normalizes the data to relative shares of the total population. This view neutralizes the effect of aggregate growth to isolate pure compositional shifts.
*   **Interactive Features:** Includes a temporal slider (1999–2024) for frame-based animation and a **Comparative Overlay** function. This allows users to superimpose a reference year (e.g., 1999) as a semi-transparent outline over the active year, enabling direct shape-based comparison.

**Analysis:** The evolution of these pyramids reveals a profound structural transformation. The "Absolute" view exposes a massive "bulge" in the 25–54 age groups starting in 2014, quantifying the surge in working-age migrant labor following gaming liberalization. However, the "Percentage" view confirms that despite this influx, the relative base of children (0–14) has contracted significantly. By 2024, the morphology shifts from a classic pyramid to a "top-heavy pillar," confirming that Macao's population growth was driven by external labor immigration while the native population continued to age rapidly—a phenomenon we define as "Dual Vulnerability."


[Open pyramid.py](sections/pyramid.py) (Population pyramids section)

[Open macao_population_pyramid.py](graphs/macao_population_pyramid.py) (Population pyramids builder)

### 4.4 Longitudinal Trends & Dual-Axis Analysis

**Objective:** To systematically analyze the correlations and divergences between population size, spatial density, social structure, and labor force composition over a 25-year trajectory.

**Visualization Description:** We implemented a **Multi-Axis Line Chart** that integrates four core dimensions into a single cohesive view:
*   **X-Axis:** Time (1999–2024), annotated with key socio-economic milestones (e.g., "Gaming Liberalization," "COVID-19").
*   **Left Y-Axis (Index):** Calibrated to a baseline (1999 = 100) to trace the relative growth trajectories of **Total Population** and **Population Density**. This index-based approach allows for direct comparison of growth rates despite differing units.
*   **Right Y-Axis (Percentage):** Displays the **Aging Ratio** and **Non-Resident Worker Ratio** (scaled 0–35%). This separates structural indicators from volume indicators to prevent scale conflicts.
*   **Visual Hierarchy:** Distinguishes variable types using distinct line styles and markers to ensure readability across overlapping trends.

**Analysis:** This chart elucidates three distinct phases of development. During the "High-Growth Period" (2002–2014), population and density indices rose in near-perfect lockstep, indicating that land reclamation efforts barely kept pace with the population influx. The "Pandemic & Recovery" phase (2020–2024) reveals a sharp volatility in the Non-Resident Worker Ratio, highlighting the region's external dependency. Crucially, the Aging Ratio demonstrates a monotonic upward trend regardless of economic fluctuations, confirming that while labor migration can temporarily dilute the aging statistic, it cannot reverse the underlying structural aging of the society.

***


[Open analysis.py](sections/analysis.py) (Analysis section)

[Open macao_demographic_trends.py](graphs/macao_demographic_trends.py) (Trends analysis graph builder)

### 4.5 Demographic Forecasting: Multi-Scenario Projections (1999–2035)

**Objective:** To transition the analysis from descriptive historical review to predictive strategic planning, employing multi-scenario modeling to assess the inevitability of structural aging under varying economic conditions through 2035.

**Visualization Description:** We constructed a **Multi-Scenario Projection Module** that extends the historical time series using linear extrapolation techniques to model three distinct policy-relevant futures. The design integrates:
*   **Scenario Pathways:**
    1.  **Baseline Scenario:** Projects future trends based on the steady-state growth average of the post-2015 adjustment period.
    2.  **High-Growth Scenario:** Assumes a return to aggressive economic expansion similar to the post-liberalization era.
    3.  **Low-Growth Scenario:** Models a systemic contraction or stagnation, reflecting potential prolonged external shocks.
*   **Uncertainty Visualization:** Utilizes semi-transparent **Prediction Bands** around the trend lines to visually communicate the confidence intervals and potential variance of the forecasts.
*   **Dual-Axis Architecture:** Retains the dashboard’s signature dual-axis format (Total Population on the left, Aging Ratio on the right) to allow for the simultaneous evaluation of population size versus structural health.

**Analysis:** The projections yield a critical, non-intuitive finding regarding the relationship between economic growth and demographics.
*   **Divergent Magnitudes:** The three scenarios paint vastly different pictures of total volume, ranging from a potential population contraction to **~650,000 (Low-Growth)** versus a surge to **~780,000 (High-Growth)**.
*   **Convergent Structure:** Despite these variances in size, the structural outcome is strikingly uniform. All three scenarios indicate that the aging ratio will continue to climb, reaching between **16.5% (High-Growth)** and **20.5% (Low-Growth)** by 2035.
*   **Core Insight:** This demonstrates that **aging is an irreversible trend** for Macao. While a "High-Growth" strategy involving massive labor importation can slightly dilute the aging ratio (by increasing the denominator), it cannot reverse the trend. Conversely, a "Low-Growth" scenario poses a systemic risk where the population contracts while the elderly proportion skyrockets, placing immense strain on social security systems. The era of "diluting aging through expansion" is effectively over, necessitating a shift in policy focus from growth to adaptation.

***


[Open analysis.py](sections/analysis.py) (Analysis section)

[Open macao_demographic_trends_forecast.py](graphs/macao_demographic_trends_forecast.py) (Forecasting graph builder)

### 4.6 Age Structure Transition: The "Growth-Based Aging" Bubble Chart

**Objective:** To synthesize twenty-five years of demographic evolution into a single four-dimensional view, explicitly visualizing the "Growth-Based Aging" paradox—the coexistence of rapid aggregate expansion and profound structural aging.

**Visualization Description:** We designed a **Four-Dimensional Bubble Chart** to map the structural journey of Macao’s society:
*   **X-Axis (Structural Balance):** Represents the net difference between the Elderly Ratio and the Children’s Ratio (Calculated as: *Elderly % minus Children %*). A value of roughly -15 indicates a young society; 0 indicates parity.
*   **Y-Axis (Youth Dependency):** Quantifies the proportion of the population aged 0–14 years.
*   **Bubble Size (Magnitude):** Scaled proportionally to the **Total Population**, allowing users to visualize physical expansion.
*   **Color Gradient (Time):** Encodes temporal progression using a continuous spectrum from **Blue (1999–2010)** to **Orange (2011–2018)** to **Red (2019–2024)**.

**Analysis:** The chart reveals a distinct three-phase morphological transformation:
1.  **Phase 1: Young Society (Blue, Upper-Left):** In the early 2000s, bubbles are clustered in the top-left quadrant (high youth ratio >16%, negative structural balance), representing a demographic surplus of children.
2.  **Phase 2: The Transition (Orange, Diagonal Drift):** Between 2011 and 2018, the bubbles migrate diagonally toward the center. This movement signifies a concurrent decline in the youth share and a narrowing gap with the elderly population.
3.  **Phase 3: The Paradox (Red, Center-Right):** In the modern era, the bubbles converge near the centerline ($x \approx 0$), indicating that the elderly and child populations have reached near parity.
*   **Key Finding:** The most critical insight is derived from the **Bubble Size**. As the bubbles shift from "Young" to "Aged," they grow substantially in diameter, reflecting a **60% increase in total population**. This visualizes the project's central thesis: Macao experienced **"Growth-Based Aging."** The massive importation of working-age labor (which drove the size increase) masked the underlying aging of the resident population but ultimately failed to prevent the structural transition. The chart confirms that Macao has successfully delayed, but not averted, the arrival of a deeply aged society, and it must now manage this reality within a context of maximum population density.

***


[Open analysis.py](sections/analysis.py) (Analysis section)

[Open bubble_chart.py](graphs/bubble_chart.py) (Bubble chart builder)

## 5. Conclusion

### 5.1 Key Findings: The Paradox of Growth-Based Aging

Our systematic analysis identifies a defining paradox in Macao’s demographic journey (1999–2024), which we characterize as "Growth-Based Aging." The data reveals that while Macao’s population expanded by nearly 60% following economic liberalization, this growth was driven almost exclusively by the importation of non-resident labor. This influx created a "demographic shield" that temporarily masked—but did not halt—the rapid, inexorable aging of the local society.

Consequently, Macao has entered a phase of structural maturity defined by a "Dual Vulnerability." The region is now highly susceptible to external labor market shocks—as dramatically evidenced by the population contraction during the pandemic—while simultaneously bearing the rising social costs of an irreversibly aging residential structure. Furthermore, the historical correlation between population growth and density suggests that the era of rapid expansion via land reclamation is diminishing. Future urban resilience will depend not on pursuing aggregate growth, but on adapting governance to these permanent spatial constraints and prioritizing the quality of life within a high-density environment.

### 5.2 Reflection on Learning

Beyond the specific demographic insights, this project served as a rigorous exercise in determining the appropriate technical approach for different stages of the data science lifecycle.

First, regarding **data engineering**, we found that the official DSEC datasets were remarkably robust and detailed, contradicting the common expectation that external data is inherently "messy." The data integrity was high, with minimal missing values that rarely required imputation. Consequently, our engineering focus shifted from "cleaning" (fixing errors) to **structural transformation**. The primary challenge lay in converting human-readable formats—specifically multi-layer hierarchical headers and wide-format tables—into machine-readable schemas suitable for analysis. We also learned the complexities of geospatial alignment, as integrating administrative vector data with statistical indicators required precise topological coordination.

Technically, we gained a nuanced understanding of our visualization stack. We discovered that **Plotly** was the essential engine for storytelling, providing the granular interactivity (such as frame-based animation and hover events) required to make the population pyramids and trend charts intuitive. Complementing this, we utilized **Streamlit** as the architectural framework. It served as the orchestration layer, managing the application state and layout—particularly in the Overview section—to bind these distinct interactive components into a cohesive, deployable web product.This distinction between visualization logic and application logic was a key technical takeaway for our team.

Ultimately, the integration of these technologies allowed us to transform abstract statistics into an accessible narrative. This experience bridged the gap between raw Python analysis and effective communication, teaching us that the value of data lies not just in its processing, but in its ability to tell a compelling story.



### 5.3 Limitations and Future Work

While our analysis provides a comprehensive historical view, we acknowledge specific constraints in data granularity and analytical scope that shaped our final deliverable.

**Data Granularity and the "Local" Perspective**

A primary ambition during the design phase was to implement a comparative feature allowing users to toggle between "Total Population" and "Local Resident Population" within the pyramid and overview modules. This would have enabled a direct visual assessment of how non-resident labor distorts the native demographic structure. However, we encountered a critical data gap: while DSEC provides robust metrics for the total population, the specific cross-tabulations required for local residents—stratified simultaneously by age group *and* gender over the full time series—were incomplete or unavailable. Consequently, we streamlined our scope to focus on the Total Population. Future work would prioritize acquiring or estimating this disaggregated resident data to enable a deeper analysis of Macao’s internal demographic health, independent of transient labor flows.

**Analytical Scope: Balancing Aging and Migration**

Furthermore, our current analytical framework is heavily weighted toward the "Growth-Based Aging" phenomenon. While this yielded critical insights into the age structure, the analysis of migration patterns and sectoral employment remains comparatively high-level. Given that non-resident workers constitute a massive portion of Macao's demographic engine, future iterations of this dashboard should reinforce the "Migration and Employment" modules. Specifically, incorporating deeper correlations between labor policy changes, specific industry demands (beyond gaming), and migration flows would provide a more holistic economic perspective.

**From Forecasting to Impact Assessment**

Finally, regarding our predictive modeling, we acknowledge that our reliance on linear extrapolation serves as a functional baseline but lacks the statistical sensitivity to capture the volatility inherent in Macao's open economy. This deterministic approach cannot account for sudden policy shifts, migration law amendments, or "Black Swan" events (such as the 2020 pandemic) that radically alter growth trajectories. Future research should prioritize upgrading this engine to incorporate stochastic models, such as **ARIMA** or **agent-based simulations**, to enable more nuanced scenario planning. Beyond improving the models themselves, a truly conclusive tool must bridge the gap between *demographics* and *infrastructure*. By integrating these refined population projections with **socio-economic impact indicators**—such as projecting healthcare demand based on aging rates or housing capacity based on density—we can transform this dashboard from a retrospective monitor into a proactive instrument for urban resilience planning.
